In [13]:
#cpi cleaner

import pandas as pd
from pandas.tseries.offsets import CustomBusinessDay

fixed_holidays = []
for y in range(2010, 2027):
    fixed_holidays.extend([
        pd.Timestamp(year=y, month=1, day=26),
        pd.Timestamp(year=y, month=8, day=15),
        pd.Timestamp(year=y, month=10, day=2),
        pd.Timestamp(year=y, month=12, day=25)
    ])

# 1. Load data — filter to All India / Combined / General headline only
cpi = pd.read_csv("cpi_220 (1).csv")
cpi['year'] = pd.to_numeric(cpi['year'], errors='coerce')
cpi['month'] = cpi['month'].astype(str).str.strip()
cpi = cpi.dropna(subset=['year', 'month'])
cpi['year'] = cpi['year'].astype(int)

# Filter to headline series which has the raw index level
cpi_headline = cpi[
    (cpi['state'] == 'All India') &
    (cpi['sector'] == 'Combined') &
    (cpi['group'] == 'General')
].copy()

# 2. Create target monthly calendar
periods = pd.period_range(start='2010-01', end='2026-05', freq='M')
df = pd.DataFrame({'Period': periods})
df['year'] = df['Period'].dt.year
df['month'] = df['Period'].dt.strftime('%B')

# Merge inflation and index level
df = pd.merge(
    df,
    cpi_headline[['year', 'month', 'index', 'inflation']].drop_duplicates(subset=['year', 'month']),
    on=['year', 'month'],
    how='left'
)

# 3. Handle Dates
df['Raw_EOM'] = df['Period'].dt.to_timestamp(how='end').dt.normalize()

def get_last_bday(dt):
    curr = dt
    while curr.weekday() >= 5 or pd.Timestamp(curr) in fixed_holidays:
        curr -= pd.Timedelta(days=1)
    return curr

def get_release_date(period):
    next_month = period + 1
    curr = pd.Timestamp(year=next_month.year, month=next_month.month, day=14)
    while curr.weekday() >= 5 or pd.Timestamp(curr) in fixed_holidays:
        curr += pd.Timedelta(days=1)
    return curr

df['EOM_Date'] = df['Raw_EOM'].apply(get_last_bday)
df['Release_Date'] = df['Period'].apply(get_release_date)

df = df[['Period', 'EOM_Date', 'Release_Date', 'index', 'inflation']].copy()
df.rename(columns={'index': 'cpi_index'}, inplace=True)
df['Period'] = df['Period'].astype(str)
df['EOM_Date'] = df['EOM_Date'].dt.strftime('%Y-%m-%d')
df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')

df.to_csv("CPI_Cleaned_Release_Dates.csv", index=False)
print(f"CPI cleaner complete. Shape: {df.shape}")
print(df[df['cpi_index'].notna()].head(10))

CPI cleaner complete. Shape: (197, 5)
     Period    EOM_Date Release_Date  cpi_index  inflation
12  2011-01  2011-01-31   2011-02-14       89.5        NaN
13  2011-02  2011-02-28   2011-03-14       88.4        NaN
14  2011-03  2011-03-31   2011-04-14       88.4        NaN
15  2011-04  2011-04-29   2011-05-16       89.1        NaN
16  2011-05  2011-05-31   2011-06-14       89.8        NaN
17  2011-06  2011-06-30   2011-07-14       90.6        NaN
18  2011-07  2011-07-29   2011-08-16       91.8        NaN
19  2011-08  2011-08-31   2011-09-14       92.7        NaN
20  2011-09  2011-09-30   2011-10-14       93.8        NaN
21  2011-10  2011-10-31   2011-11-14       94.8        NaN


In [14]:
#wpi cleaner

import pandas as pd
import numpy as np

fixed_holidays = []
for y in range(2010, 2027):
    fixed_holidays.extend([
        pd.Timestamp(year=y, month=1, day=26),
        pd.Timestamp(year=y, month=8, day=15),
        pd.Timestamp(year=y, month=10, day=2),
        pd.Timestamp(year=y, month=12, day=25)
    ])

# 1. Load raw file with no header parsing
wpi_raw = pd.read_csv("Wholesale Price Index - Monthly (Base _ 2011-12 = 100).csv", header=None)

# All Commodities data is in rows 8-22 (row 7 is the header)
# Column 1 = FY label, Columns 2-13 = APR through MAR
month_cols = ['APR.', 'MAY', 'JUN.', 'JUL.', 'AUG.', 'SEP.', 'OCT.', 'NOV.', 'DEC.', 'JAN.', 'FEB.', 'MAR.']
month_num_map = {'APR.': 4, 'MAY': 5, 'JUN.': 6, 'JUL.': 7,
                 'AUG.': 8, 'SEP.': 9, 'OCT.': 10, 'NOV.': 11,
                 'DEC.': 12, 'JAN.': 1, 'FEB.': 2, 'MAR.': 3}

# Rows 8 to 22 are All Commodities data rows
data_rows = wpi_raw.iloc[8:23].copy().reset_index(drop=True)

records = []
for _, row in data_rows.iterrows():
    fy = str(row[1]).strip()
    if not fy or fy == 'nan':
        continue
    try:
        start_year = int(fy.split('-')[0])
    except:
        continue
    for col_idx, month_name in enumerate(month_cols):
        val = row[col_idx + 2]
        if pd.isna(val) or str(val).strip() == '' or str(val).strip() == 'nan':
            continue
        try:
            wpi_index = float(str(val).strip())
        except:
            continue
        month_num = month_num_map[month_name]
        cal_year = start_year if month_num >= 4 else start_year + 1
        records.append({
            'Period_DT': pd.Timestamp(year=cal_year, month=month_num, day=1),
            'wpi_index': wpi_index
        })

df = pd.DataFrame(records).sort_values('Period_DT').reset_index(drop=True)

# 2. Compute WPI YoY inflation from the index level
df['wpi_inflation'] = df['wpi_index'].pct_change(12) * 100

# 3. Replace any zero values with NaN — zeros indicate missing/padding, not real data
# WPI source starts April 2012, so anything before is not real
df.loc[df['wpi_index'] == 0, 'wpi_index'] = np.nan
df.loc[df['wpi_inflation'] == 0, 'wpi_inflation'] = np.nan

# Also explicitly null out pre-April 2012 rows since source doesn't cover them
df.loc[df['Period_DT'] < pd.Timestamp('2012-04-01'), 'wpi_index'] = np.nan
df.loc[df['Period_DT'] < pd.Timestamp('2012-04-01'), 'wpi_inflation'] = np.nan

# YoY needs 12 months of index history, so first valid YoY is April 2013
df.loc[df['Period_DT'] < pd.Timestamp('2013-04-01'), 'wpi_inflation'] = np.nan

# 4. Date rollover logic
df['Raw_EOM'] = df['Period_DT'] + pd.offsets.MonthEnd(0)

def get_last_bday(dt):
    curr = dt
    while curr.weekday() >= 5 or pd.Timestamp(curr) in fixed_holidays:
        curr -= pd.Timedelta(days=1)
    return curr

def get_release_date(period_dt):
    next_month = period_dt + pd.DateOffset(months=1)
    curr = pd.Timestamp(year=next_month.year, month=next_month.month, day=14)
    while curr.weekday() >= 5 or pd.Timestamp(curr) in fixed_holidays:
        curr += pd.Timedelta(days=1)
    return curr

df['EOM_Date'] = df['Raw_EOM'].apply(get_last_bday)
df['Release_Date'] = df['Period_DT'].apply(get_release_date)
df['Period'] = df['Period_DT'].dt.strftime('%Y-%m')
df['EOM_Date'] = df['EOM_Date'].dt.strftime('%Y-%m-%d')
df['Release_Date'] = df['Release_Date'].dt.strftime('%Y-%m-%d')

final_df = df[['Period', 'EOM_Date', 'Release_Date', 'wpi_index', 'wpi_inflation']].copy()
final_df.to_csv("WPI_Cleaned_Release_Dates.csv", index=False)
print(f"WPI cleaner complete. Shape: {final_df.shape}")
print(final_df.head(20))
print(f"\nFirst valid wpi_inflation: {final_df.dropna(subset=['wpi_inflation'])['Period'].iloc[-1]}")
print(f"Total non-null wpi_inflation rows: {final_df['wpi_inflation'].notna().sum()}")

WPI cleaner complete. Shape: (169, 5)
     Period    EOM_Date Release_Date  wpi_index  wpi_inflation
0   2012-04  2012-04-30   2012-05-14      104.7            NaN
1   2012-05  2012-05-31   2012-06-14      105.3            NaN
2   2012-06  2012-06-29   2012-07-16      105.3            NaN
3   2012-07  2012-07-31   2012-08-14      106.2            NaN
4   2012-08  2012-08-31   2012-09-14      106.9            NaN
5   2012-09  2012-09-28   2012-10-15      107.6            NaN
6   2012-10  2012-10-31   2012-11-14      107.4            NaN
7   2012-11  2012-11-30   2012-12-14      107.3            NaN
8   2012-12  2012-12-31   2013-01-14      107.1            NaN
9   2013-01  2013-01-31   2013-02-14      108.0            NaN
10  2013-02  2013-02-28   2013-03-14      108.4            NaN
11  2013-03  2013-03-29   2013-04-15      108.6            NaN
12  2013-04  2013-04-30   2013-05-14      108.6       3.724928
13  2013-05  2013-05-31   2013-06-14      108.6       3.133903
14  2013-06  2013

In [15]:
import pandas as pd

def build_inflation_pillar(cpi_csv_path, wpi_csv_path, output_csv_path):
    cpi_clean = pd.read_csv(cpi_csv_path)
    wpi_clean = pd.read_csv(wpi_csv_path)

    cpi_clean['Release_Date_DT'] = pd.to_datetime(cpi_clean['Release_Date'])
    wpi_clean['Release_Date_DT'] = pd.to_datetime(wpi_clean['Release_Date'])

    timeline_dates = pd.to_datetime(cpi_clean['EOM_Date'].dropna().unique())
    timeline_df = pd.DataFrame({'As_Of_EOM_Date': timeline_dates}).sort_values('As_Of_EOM_Date').reset_index(drop=True)

    cpi_sorted = cpi_clean.dropna(subset=['inflation']).sort_values('Release_Date_DT').reset_index(drop=True)
    wpi_sorted = wpi_clean.dropna(subset=['wpi_inflation']).sort_values('Release_Date_DT').reset_index(drop=True)

    # Carry both index levels through for the COVID fix
    merged_pillar = pd.merge_asof(
        timeline_df,
        cpi_sorted[['Release_Date_DT', 'Period', 'cpi_index', 'inflation']],
        left_on='As_Of_EOM_Date',
        right_on='Release_Date_DT',
        direction='backward'
    )
    merged_pillar.rename(columns={'Period': 'cpi_data_period', 'inflation': 'cpi_headline_inflation'}, inplace=True)
    merged_pillar.drop(columns=['Release_Date_DT'], inplace=True)

    merged_pillar = pd.merge_asof(
        merged_pillar,
        wpi_sorted[['Release_Date_DT', 'Period', 'wpi_index', 'wpi_inflation']],
        left_on='As_Of_EOM_Date',
        right_on='Release_Date_DT',
        direction='backward'
    )
    merged_pillar.rename(columns={'Period': 'wpi_data_period', 'wpi_inflation': 'wpi_headline_inflation'}, inplace=True)
    merged_pillar.drop(columns=['Release_Date_DT'], inplace=True)

    merged_pillar['wpi_cpi_spread'] = merged_pillar['wpi_headline_inflation'] - merged_pillar['cpi_headline_inflation']

    merged_pillar = merged_pillar.sort_values(by='As_Of_EOM_Date', ascending=False).reset_index(drop=True)
    merged_pillar['As_Of_EOM_Date'] = merged_pillar['As_Of_EOM_Date'].dt.strftime('%Y-%m-%d')
    merged_pillar.dropna(subset=['cpi_headline_inflation', 'wpi_headline_inflation'], how='all', inplace=True)

    merged_pillar.to_csv(output_csv_path, index=False)
    print(f"Success! Merged pillar saved to {output_csv_path}, shape: {merged_pillar.shape}")
    return merged_pillar

merged_df = build_inflation_pillar(
    cpi_csv_path='CPI_Cleaned_Release_Dates.csv',
    wpi_csv_path='WPI_Cleaned_Release_Dates.csv',
    output_csv_path='Inflation_Pillar_Merged_Data.csv'
)

Success! Merged pillar saved to Inflation_Pillar_Merged_Data.csv, shape: (172, 8)


In [ ]:
import pandas as pd
import numpy as np

def generate_hybrid_inflation_signals(merged_pillar_csv, output_csv):
    df = pd.read_csv(merged_pillar_csv)
    df['As_Of_EOM_Date'] = pd.to_datetime(df['As_Of_EOM_Date'])
    df = df.sort_values('As_Of_EOM_Date', ascending=True).reset_index(drop=True)

    # --- COVID BASE-EFFECT FIX: 2-Year Stacked CAGR ---
    # Applied to both CPI and WPI using raw index levels
    # For any month M where M-12 falls in Mar 2020-Sep 2021,
    # replace YoY with (Index_M / Index_M-24)^0.5 - 1, annualized over 2 years
    dist_start = pd.Timestamp('2020-03-01')
    dist_end = pd.Timestamp('2021-09-01')

    def apply_2y_cagr(df, index_col, inflation_col):
        fixed = df[inflation_col].copy()
        for i in range(24, len(df)):
            month_minus_12 = df['As_Of_EOM_Date'].iloc[i] - pd.DateOffset(months=12)
            if dist_start <= month_minus_12 <= dist_end:
                val_m = df[index_col].iloc[i]
                val_m24 = df[index_col].iloc[i - 24]
                if pd.notna(val_m) and pd.notna(val_m24) and val_m24 != 0 and val_m / val_m24 > 0:
                    fixed.iloc[i] = ((val_m / val_m24) ** 0.5 - 1) * 100
                else:
                    fixed.iloc[i] = np.nan
        return fixed

    df['cpi_headline_inflation'] = apply_2y_cagr(df, 'cpi_index', 'cpi_headline_inflation')
    df['wpi_headline_inflation'] = apply_2y_cagr(df, 'wpi_index', 'wpi_headline_inflation')

    # Recompute spread after both fixes
    df['wpi_cpi_spread'] = df['wpi_headline_inflation'] - df['cpi_headline_inflation']

    # --- ROBUST Z-SCORE ENGINE ---
    def calc_robust_z_winsorized(window_slice):
        current_val = window_slice[-1]
        if np.isnan(current_val):
            return np.nan
        valid_vals = window_slice[~np.isnan(window_slice)]
        if len(valid_vals) == 0:
            return np.nan
        med = np.median(valid_vals)
        mad = np.median(np.abs(valid_vals - med))
        if mad == 0:
            z = 0.0
        else:
            z = (current_val - med) / (1.4826 * mad)
        return np.clip(z, -4.0, 4.0)

    df['cpi_inflation'] = df['cpi_headline_inflation'].rolling(window=36, min_periods=18).apply(calc_robust_z_winsorized, raw=True)
    df['wpi_inflation'] = df['wpi_headline_inflation'].rolling(window=36, min_periods=18).apply(calc_robust_z_winsorized, raw=True)
    df['wpi_cpi_spread'] = df['wpi_cpi_spread'].rolling(window=36, min_periods=18).apply(calc_robust_z_winsorized, raw=True)

    df_final = df.sort_values('As_Of_EOM_Date', ascending=False).reset_index(drop=True)
    df_final['date'] = df_final['As_Of_EOM_Date'].dt.strftime('%Y-%m-%d')

    target_cols = ['date', 'cpi_inflation', 'wpi_inflation', 'wpi_cpi_spread']
    df_final = df_final[target_cols]
    df_final.dropna(subset=['cpi_inflation', 'wpi_inflation'], how='all', inplace=True)

    df_final.to_csv(output_csv, index=False)
    print(f"Success! Saved to {output_csv}. Total Rows: {df_final.shape[0]}")
    return df_final

clean_signals_df = generate_hybrid_inflation_signals(
    merged_pillar_csv='Inflation_Pillar_Merged_Data.csv',
    output_csv='Inflation_Pillar_With_Signals.csv'
)

Success! Saved to Inflation_Pillar_With_Signals.csv. Total Rows: 145


In [17]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def format_inflation_workbook(input_csv, output_xlsx):

    df = pd.read_csv(input_csv)

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Inflation Signals"
    ws.views.sheetView[0].showGridLines = True

    fill_extreme_high = PatternFill(start_color="ffcccc", end_color="ffcccc", fill_type="solid")
    fill_mild_high    = PatternFill(start_color="ffe6e6", end_color="ffe6e6", fill_type="solid")
    fill_extreme_low  = PatternFill(start_color="ccffcc", end_color="ccffcc", fill_type="solid")
    fill_mild_low     = PatternFill(start_color="e6ffe6", end_color="e6ffe6", fill_type="solid")

    font_extreme_high = Font(name="Calibri", size=11, color="990000", bold=True)
    font_mild_high    = Font(name="Calibri", size=11, color="990000", bold=False)
    font_extreme_low  = Font(name="Calibri", size=11, color="006600", bold=True)
    font_mild_low     = Font(name="Calibri", size=11, color="006600", bold=False)
    font_normal       = Font(name="Calibri", size=11, color="000000", bold=False)

    header_fill  = PatternFill(start_color="1f497d", end_color="1f497d", fill_type="solid")
    header_font  = Font(name="Calibri", size=11, color="ffffff", bold=True)
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

    thin_border = Border(
        left=Side(style='thin',   color='d9d9d9'),
        right=Side(style='thin',  color='d9d9d9'),
        top=Side(style='thin',    color='d9d9d9'),
        bottom=Side(style='thin', color='d9d9d9')
    )

    target_cols = ['date', 'cpi_inflation', 'wpi_inflation', 'wpi_cpi_spread']
    ws.append(target_cols)
    for col_idx in range(1, 5):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = header_align

    for _, row in df.iterrows():
        row_values = [row['date'], row['cpi_inflation'], row['wpi_inflation'], row['wpi_cpi_spread']]
        ws.append(row_values)
        current_row = ws.max_row

        d_cell = ws.cell(row=current_row, column=1)
        d_cell.alignment = Alignment(horizontal="center")
        d_cell.font = font_normal
        d_cell.border = thin_border

        for col_idx in range(2, 5):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.number_format = '0.0000'
            cell.alignment = Alignment(horizontal="right")
            cell.border = thin_border

            val = row_values[col_idx - 1]
            if pd.isna(val):
                cell.font = font_normal
                continue

            if val >= 2.0:
                cell.fill = fill_extreme_high
                cell.font = font_extreme_high
            elif 1.0 <= val < 2.0:
                cell.fill = fill_mild_high
                cell.font = font_mild_high
            elif val <= -2.0:
                cell.fill = fill_extreme_low
                cell.font = font_extreme_low
            elif -2.0 < val <= -1.0:
                cell.fill = fill_mild_low
                cell.font = font_mild_low
            else:
                cell.font = font_normal

    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 14)

    wb.save(output_xlsx)
    print(f"Success! Highlighted spreadsheet saved to {output_xlsx}")

format_inflation_workbook(
    input_csv='Inflation_Pillar_With_Signals.csv',
    output_xlsx='Inflation_Pillar_Scores_Highlighted.xlsx'
)

Success! Highlighted spreadsheet saved to Inflation_Pillar_Scores_Highlighted.xlsx


In [18]:
import pandas as pd
import numpy as np
import os
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================================
# 1. STANDARDIZED 5-POINT SCORECARD GENERATOR
# =========================================================================
def map_to_5_point_scale(z_score):
    """
    Standardizes continuous variables onto a balanced 5-point scale [-1.0 to 1.0].
    Prevents volatile cliff-edge jumps by grouping data into standard deviation bands.
    """
    if pd.isna(z_score): 
        return 0.0
    if z_score >= 1.2: 
        return 1.0     # Strongly Stressed / High Vulnerability
    elif z_score >= 0.8: 
        return 0.75     # Moderately Stressed
    elif z_score >= 0.45: 
        return 0.5     # Moderately Stressed
    elif z_score >= 0.15: 
        return 0.25     # Moderately Stressed
    elif z_score <= -1.5: 
        return -1.0    # Strongly Favorable / High Cushion
    elif z_score <= -1.1: 
        return -0.75    # Strongly Favorable / High Cushion
    elif z_score <= -0.65: 
        return -0.5    # Strongly Favorable / High Cushion
    elif z_score <= -0.35: 
        return -0.25    # Moderately Favorable
    else: 
        return 0.0     # Neutral Anchor    

def classify_net_to_7_point_regime(net_score):
    """
    Translates a continuous composite net score [-1.0 to 1.0] 
    into a stable, balanced 7-point absolute macro regime scale.
    """
    if pd.isna(net_score):   return 0.0
    if net_score >= 0.70:    return 3.0    # Extreme Tail Acceleration
    elif net_score >= 0.45:  return 2.0    # Strong Acceleration
    elif net_score >= 0.15:  return 1.0    # Mild/Orderly Upward Trend
    elif net_score <= -0.70: return -3.0   # Extreme Tail Collapse
    elif net_score <= -0.45: return -2.0   # Strong Deceleration
    elif net_score <= -0.15: return -1.0   # Mild/Orderly Downward Trend
    else:                    return 0.0    # Absolute Structural Neutral

# =========================================================================
# 2. HIERARCHICAL PILLAR CALCULATOR (CPI Anchor + Wholesale Modifier)
# =========================================================================
def calculate_hierarchical_inflation_score(cpi_z, wpi_z, spread_z):
    """
    Calculates a composite inflation score using a micro-hierarchical architecture:
    - CPI is the primary net anchor (dictating the foundational baseline regime).
    - WPI and the Spread act as upstream transmission gears that either amplify 
      conviction (when aligned) or inject structural friction/drags (when dislocated).
    """
    if pd.isna(cpi_z):
        return np.nan
        
    cpi_f = map_to_5_point_scale(cpi_z)
    wpi_f = map_to_5_point_scale(wpi_z) if pd.notna(wpi_z) else 0.0
    
    # Establish baseline anchor
    base_score = cpi_f
    
    # Determine downstream transmission behavior
    if cpi_f == 0.0:
        # Case A: Consumer baseline is neutral; check if WPI is early-warning leading
        transmission_modifier = 0.25 * wpi_f
        adjusted_score = base_score + transmission_modifier
    elif cpi_f * wpi_f > 0:
        # Case B: Alignment/Amplification (Both indices moving in the same macro vector)
        amplification = 0.10 * wpi_f
        adjusted_score = base_score + amplification
    else:
        # Case C: Dislocation / Cross-Currents (e.g., Supply Shock vs Margin Squeezes)
        dislocation_modifier = 0.50 * (wpi_f - cpi_f)
        adjusted_score = base_score + dislocation_modifier

    # Guarantee rigorous boundary bounds
    final_net_score = max(-1.0, min(1.0, adjusted_score))
    return final_net_score

# =========================================================================
# 3. TIME-SERIES TIMING ENGINE (UNIFIED LOOKBACK & MEDIAN FALLBACK)
# =========================================================================
def apply_asymmetric_lookback_filter(regimes):
    """
    Advanced Chronological Timing Engine.
    Implements a 'Fast-Attack, Slow-Decay' asymmetric gate.
    - Fast Attack: Maximum tail shocks (+3 or -3) bypass the window and print instantly.
    - Slow Decay: Protects the portfolio from immediate structural whipsaws by 
      forcing a controlled step-down when exiting a severe crisis state.
    """
    confirmed = []
    current_confirmed = 0.0
    
    for i in range(len(regimes)):
        flash = regimes[i]
        
        if i < 2:
            current_confirmed = flash if pd.notna(flash) else 0.0
            confirmed.append(current_confirmed)
            continue
            
        # 1. FAST ATTACK GATE
        if flash == 3.0 or flash == -3.0:
            current_confirmed = flash
            confirmed.append(current_confirmed)
            continue
            
        # 2. STANDARD FILTER PROCESSING (2-out-of-3 month confirmation rule)
        window = [regimes[i], regimes[i-1], regimes[i-2]]
        window = [v for v in window if pd.notna(v)]
        
        if len(window) == 0:
            confirmed.append(current_confirmed)
            continue
            
        counts = pd.Series(window).value_counts()
        highest_frequency = counts.iloc[0]
        most_frequent_value = counts.index[0]
        
        if highest_frequency >= 2:
            proposed_state = most_frequent_value
        else:
            proposed_state = float(np.median(window))
            
        # 3. SLOW DECAY GATE (Hysteresis Loop cushion floor)
        if current_confirmed == 3.0 and proposed_state < 1.0:
            current_confirmed = 1.0  
        elif current_confirmed == -3.0 and proposed_state > -1.0:
            current_confirmed = -1.0 
        else:
            current_confirmed = proposed_state
            
        confirmed.append(current_confirmed)
        
    return confirmed

# =========================================================================
# 4. WORKBOOK EXCEL PROCESSING & VISUAL PRESENTATION PIPELINE
# =========================================================================
def process_inflation_sheet(input_file, output_file):
    """
    Reads inflation datasets, runs processing pipelines, implements lookback
    confirmation, and formats output with institutional styling architectures.
    """
    if not os.path.exists(input_file):
        print(f"Error: Could not locate input file '{input_file}' in your workspace.")
        return

    df = pd.read_excel(input_file)
    date_col = 'date' if 'date' in df.columns else 'Last Day of Month'
    if date_col not in df.columns:
        raise KeyError(f"Could not find valid date identifier. Columns found: {list(df.columns)}")

    # Step 1: Sort chronologically for timing engine lookback integrity
    df['_chrono_helper'] = pd.to_datetime(df[date_col])
    df = df.sort_values(by='_chrono_helper', ascending=True).reset_index(drop=True)
    
    # Step 2: Calculate underlying smooth net scores using hierarchical matrix logic
    df['Net_Composite_Score'] = df.apply(
        lambda r: calculate_hierarchical_inflation_score(r['cpi_inflation'], r['wpi_inflation'], r['wpi_cpi_spread']), 
        axis=1
    )
    flash_regimes = [classify_net_to_7_point_regime(val) for val in df['Net_Composite_Score']]
    df['Raw_Inflation_Regime'] = flash_regimes
    
    # Step 3: Run the 2-out-of-3 month filter with Median Fallback
    df['Confirmed_Inflation_Regime'] = apply_asymmetric_lookback_filter(flash_regimes)
    
    # Step 4: Revert to presentation sorting layout (latest dates on top)
    df_desc = df.sort_values(by='_chrono_helper', ascending=False).reset_index(drop=True)
    df_desc = df_desc.drop(columns=['_chrono_helper'])
    
    # Step 5: Save directly to basic Excel layout
    df_desc.to_excel(output_file, index=False, sheet_name="Confirmed Inflation")
    
    # Step 6: Post-Processing openpyxl Polish Engine
    wb = openpyxl.load_workbook(output_file)
    ws = wb.active
    ws.views.sheetView[0].showGridLines = True
    
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )
    
    for col_idx in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border
        
        col_letter = openpyxl.utils.get_column_letter(col_idx)
        ws.column_dimensions[col_letter].width = 24
        
        for row_idx in range(2, ws.max_row + 1):
            data_cell = ws.cell(row=row_idx, column=col_idx)
            data_cell.border = thin_border
            
            if isinstance(data_cell.value, (int, float)):
                data_cell.alignment = Alignment(horizontal="right")
                if "Regime" in str(cell.value) or "Score" in str(cell.value):
                    data_cell.number_format = '0.00'
            elif isinstance(data_cell.value, pd.Timestamp) or "datetime" in str(type(data_cell.value)):
                data_cell.alignment = Alignment(horizontal="center")
                data_cell.number_format = 'yyyy-mm-dd'

    wb.save(output_file)
    print(f"Process complete! Standardized output safely compiled to: '{output_file}'")
    print("\nFinal Smooth Gated Distribution:")
    print(df_desc['Confirmed_Inflation_Regime'].value_counts().sort_index())

# =========================================================================
# 5. ENTRYPOINT RUNNER
# =========================================================================
if __name__ == "__main__":
    INFLATION_SOURCE = "Inflation_Pillar_Scores_Highlighted.xlsx" 
    INFLATION_OUTPUT = "Output_Confirmed_Smooth_Inflation.xlsx"
    
    process_inflation_sheet(input_file=INFLATION_SOURCE, output_file=INFLATION_OUTPUT)

Process complete! Standardized output safely compiled to: 'Output_Confirmed_Smooth_Inflation.xlsx'

Final Smooth Gated Distribution:
Confirmed_Inflation_Regime
-3.0    27
-2.0    15
-1.0    28
 0.0    39
 1.0    23
 2.0     9
 3.0     4
Name: count, dtype: int64
